In [ ]:
import pandas as pd
import json 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Multiple Dataset

In [ ]:
representative_order = [
    #Popular Representation
    'pipe_serialized',
    'token_serialized',
    'space_serialized',
    # Data Representation
    'csv',
    'tsv',
    'html',
    'markdown',
    'latex',
    'dict',
    'json',
    'xml',
    # Structural Transformations 
    'shuffled_rows',
    'shuffled_cols',
    'transpose',
    #Schema Definition Types
    'mschema',
    'macschema',
    'ddl',
    #Centroid
    "centroid_popular",
    "centroid_data",
    "centroid_schema",
    "centroid_structural",
     "centroid_all",
]
category = {"Popular Representation": ['pipe_serialized',    'token_serialized', 'space_serialized',"centroid_popular"],
    "Data Representation" : [
            'csv',
            'tsv',
            'html',
            'markdown',
            'latex',
            'dict',
            'json',
            'xml',
            'centroid_data'],
    "Structural Transformations": ['shuffled_rows',
        'shuffled_cols',
        'transpose',
        'centroid_structural'],
    "Schema Definition Types" :['mschema',
    'macschema',
    'ddl',
    'centroid_schema'],
    "All" :['centroid_all'],
}
category_wo_centroid = {"Popular Representation": ['pipe_serialized',    'token_serialized', 'space_serialized'],
    "Data Representation" : [
            'csv',
            'tsv',
            'html',
            'markdown',
            'latex',
            'dict',
            'json',
            'xml',
            ],
    "Structural Transformations": ['shuffled_rows',
        'shuffled_cols',
        'transpose'],
    "Schema Definition Types" :['mschema',
    'macschema',
    'ddl'],
}

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_list = []
representative_wo_order = [
    #Popular Representation
    'pipe_serialized','token_serialized','space_serialized',
    # Data Representation
    'csv','tsv','html','markdown','latex','dict','json','xml',
    # Structural Transformations 
    'shuffled_rows','shuffled_cols','transpose',
    #Schema Definition Types
    'mschema','macschema','ddl',
]
for adapter_type in ["base", "adapter", "adapter_subset"]:
    for dataset in ["WTQ", "WIKISQL", "NQ"]:
        for model in ["splade", "mpnet", "bge", "reasonir"]:
            model_train_date = "2026-03-10" if model != "splade" else "2026-03-01"
            df_model = None

            for idx, representative in enumerate(representative_wo_order):
                if adapter_type == "base":
                    fpath = (
                        f"./data/retrieval_all/"
                        f"{model}_results_rank/{dataset}/{representative}_gold_rank_per_question.csv"
                    )
                else:
                    fpath = (
                        f"./data/retrieval_all/"
                        f"{model}_results_rank_with_{adapter_type}/{model_train_date}/"
                        f"{dataset}/{representative}_gold_rank_per_question.csv"
                    )

                df = pd.read_csv(fpath)
                df = df[["model", "dataset", "question_id", "rank", "hit@1"]]

                df_renamed = df.rename(
                    columns={
                        "rank": representative,
                        "hit@1": f"{representative}_hit@1",
                    }
                )

                if idx == 0:
                    df_model = df_renamed
                else:
                    df_model = df_model.merge(
                        df_renamed,
                        on=["model", "dataset", "question_id"],
                        how="inner",
                        validate="one_to_one",
                    )

            df_model["adapter_type"] = adapter_type
            df_list.append(df_model)

df = pd.concat(df_list, ignore_index=True)
df = df.drop_duplicates()

# Comparison

In [ ]:
import pandas as pd
import numpy as np

id_cols = ["model", "dataset", "question_id", "adapter_type"]

rank_cols = [
    c for c in df.columns
    if c not in id_cols and not c.endswith("_hit@1")
]
hit_cols = [f"{c}_hit@1" for c in rank_cols]

rank_long = df.melt(
    id_vars=id_cols,
    value_vars=rank_cols,
    var_name="representation",
    value_name="rank",
)

hit_long = df.melt(
    id_vars=id_cols,
    value_vars=hit_cols,
    var_name="representation_hit",
    value_name="hit1",
)

hit_long["representation"] = hit_long["representation_hit"].str.replace("_hit@1", "", regex=False)
hit_long = hit_long.drop(columns=["representation_hit"])

long_df = rank_long.merge(
    hit_long,
    on=["model", "dataset", "question_id", "adapter_type", "representation"],
    how="inner",
    validate="one_to_one",
)

base_df = (
    long_df[long_df["adapter_type"] == "base"]
    .rename(columns={"rank": "base_rank", "hit1": "base_hit1"})
    .drop(columns=["adapter_type"])
)

cmp_list = []

for comp in ["adapter", "adapter_subset"]:
    adapt_sub = (
        long_df[long_df["adapter_type"] == comp]
        .rename(columns={"rank": "adapt_rank", "hit1": "adapt_hit1"})
        .drop(columns=["adapter_type"])
    )

    cmp_sub = base_df.merge(
        adapt_sub,
        on=["model", "dataset", "question_id", "representation"],
        how="inner",
        validate="one_to_one",
    )

    cmp_sub["compare_to"] = comp
    cmp_list.append(cmp_sub)

cmp = pd.concat(cmp_list, ignore_index=True)

cmp["delta_rank"] = cmp["base_rank"] - cmp["adapt_rank"]
cmp["improved"] = cmp["adapt_rank"] < cmp["base_rank"]
cmp["worsened"] = cmp["adapt_rank"] > cmp["base_rank"]
cmp["same"] = cmp["adapt_rank"] == cmp["base_rank"]
cmp["delta_hit1"] = cmp["adapt_hit1"] - cmp["base_hit1"]

In [ ]:
representative_order = [
    # Popular Representation
    'pipe_serialized',
    'token_serialized',
    'space_serialized',
    # Data Representation
    'csv',
    'tsv',
    'html',
    'markdown',
    'latex',
    'dict',
    'json',
    'xml',
    # Structural Transformations
    'shuffled_rows',
    'shuffled_cols',
    'transpose',
    # Schema Definition Types
    'mschema',
    'macschema',
    'ddl',
]
model_order = ["reasonir", "mpnet", "bge", "splade"]

def draw_heatmap(ax, pivot, title, cmap, vmin=None, vmax=None, fmt="{:.2f}", show_yticks=True):
    im = ax.imshow(pivot.values, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(pivot.columns)))
    #ax.set_xticklabels(pivot.columns, rotation=45, ha="right", fontsize=16)
    ax.set_yticks(range(len(pivot.index)))
    if show_yticks:
        ax.set_yticklabels(pivot.index, fontsize=16)
    else:
        ax.set_yticklabels([])
    ax.set_title(title)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.iloc[i, j]
            if pd.notna(val):
                ax.text(j, i, fmt.format(val), ha="center", va="center", fontsize=12)
    return im

for comp in ["adapter", "adapter_subset"]:
    fig, axes = plt.subplots(1, len(cmp["dataset"].unique()), figsize=(20, 6))
    N = len(cmp["dataset"].unique())
    for idx, dataset in enumerate(sorted(cmp["dataset"].unique(), reverse=True)):
        sub = cmp[(cmp["compare_to"] == comp) & (cmp["dataset"] == dataset)]
        delta_pivot = (
            sub.groupby(["representation", "model"])["log_delta_rank"]
            .mean()
            .unstack("model")
        )
        # Reindex rows according to representative_order, keeping only those present
        ordered_index = [r for r in representative_order if r in delta_pivot.index]
        delta_pivot = delta_pivot.reindex(ordered_index)
        # Reindex columns  ← add this
        ordered_cols = [m for m in model_order if m in delta_pivot.columns]
        delta_pivot = delta_pivot.reindex(columns=ordered_cols)
        win_pivot = win_pivot.reindex(columns=ordered_cols)
        win_pivot = (
            sub.groupby(["representation", "model"])["improved"]
            .mean()
            .unstack("model")
        )
        win_pivot = win_pivot.reindex(ordered_index)

        v = np.nanpercentile(np.abs(delta_pivot.values), 95)

        show_yticks = (idx == -1)  # Only show y-tick labels on the leftmost subplot
        im1 = draw_heatmap(
            axes[idx],
            delta_pivot,
            title=f"{dataset}",#{comp.capitalize().replace('_subset', ' Subset')} vs Base, 
            cmap="coolwarm",
            vmin=-v,
            vmax=v,
            fmt="{:.2f}",
            show_yticks=show_yticks,
        )
        axes[idx].set_title(
            f"{dataset}",#{comp.capitalize().replace('_subset', ' Subset')} vs Base, 
            fontsize=16
        )
        axes[idx].tick_params(axis='x', labelsize=16)
        axes[idx].set_xlabel(axes[idx].get_xlabel(), fontsize=16)

        cbar = fig.colorbar(im1, ax=axes[idx])
        cbar.ax.tick_params(labelsize=16)
        if idx == N - 1:
            cbar.set_label("Δ log-rank (base - adapted)", fontsize=20)

    plt.tight_layout()
    plt.show()